From `16_notebook_refactor.ipynb`

In [1]:
import dagshub
import dill as pickle
import joblib
import mlflow
from mlflow.models import infer_signature

# import pandas as pd
import polars as pl
import re
from sklearn.feature_extraction.text import (
    CountVectorizer,
    TfidfTransformer,
    TfidfVectorizer,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
import stanza
from typing import Dict, Text, List
from tqdm import tqdm
import sys
from pathlib import Path

# Adds the immediate parent directory to the runtime path
sys.path.insert(0, str(Path.cwd().parent))

from src.backend.embedding_creation.apply_stanza import CustomSKLearnAnalyzer
from src.backend.embedding_creation.sklearn_transformer_as_mlflow_model import (
    CustomSKLearnWrapper,
)

## Need to call DAGsHub to keep track of what we're doing

In [2]:
# @markdown Enter the username of your DAGsHub account:
DAGSHUB_USER_NAME = "AaronWChen"  # @param {type:"string"}

# @markdown Enter the email for your DAGsHub account:
DAGSHUB_EMAIL = "awc33@cornell.edu"  # @param {type:"string"}

# @markdown Enter the repo name
DAGSHUB_REPO_NAME = "MeaLeon"

# @markdown Enter the name of the branch you are working on
BRANCH = "20260710/new_tfidf"
dagshub.init(repo_name=DAGSHUB_REPO_NAME, repo_owner=DAGSHUB_USER_NAME)

Accessing as AaronWChen

Initialized MLflow to track repo "AaronWChen/MeaLeon"

Repository AaronWChen/MeaLeon initialized!

In [3]:
# raw data

!dvc add "../data/recipes-en-201706/ar_bbc_cookstr_epic_combined_to_vespa.json"
combined_df = pl.read_json(
    "../data/recipes-en-201706/ar_bbc_cookstr_epic_combined_to_vespa.json"
)

 ⠋ Checking graph
Adding...                                                                       
!
                                                                                
!
                                                                                
!
  0% Checking cache in '/home/awchen/Repos/Projects/MeaLeon/.dvc/cache/files/md5
                                                                                
!
  0% Querying remote cache|                          |0/1 [00:00<?,    ?files/s]
100% Querying remote cache|█████████████████████|1/1 [00:01<00:00,  1.35s/files]
                                                                                
!
  0%|          |Fetching from s3://dvc/files/md5      0/1 [00:00<?,     ?file/s]
  0%|          |Fetching from s3://dvc/files/md5      0/1 [00:00<?,     ?file/s]

!

  0%|          |dvc/files/md5/de/c8e25c5144cdd10.00/3.85k [00:00<?,        ?B/s]

100%|██████████|dvc/files/md5/de/c8e25c5143.85k/3.85k [00:00<00:00,    28.6k

In [3]:
# instantiate stanza pipeline
stanza.download("en")
nlp = stanza.Pipeline(
    "en",
    depparse_batch_size=50,
    depparse_min_length_to_batch_separately=50,
    verbose=True,
    use_gpu=True,  # set to true when on cloud/not on streaming computer
    batch_size=100,
)

2026-07-16 09:42:56 INFO: Downloaded file to /home/awchen/stanza_resources/resources.json
2026-07-16 09:42:56 INFO: Downloading default packages for language: en (English) ...
2026-07-16 09:42:57 INFO: File exists: /home/awchen/stanza_resources/en/default.zip
2026-07-16 09:43:02 INFO: Finished downloading models and saved to /home/awchen/stanza_resources
2026-07-16 09:43:02 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2026-07-16 09:43:02 INFO: Downloaded file to /home/awchen/stanza_resources/resources.json
2026-07-16 09:43:03 INFO: Loading these models for language: en (English):
| Processor    | Package                   |
--------------------------------------------
| tokenize     | combined                  |
| mwt          | combined                  |
| pos          | combined_charlm           |
| lemma        | combined_nocharlm         |
| constituency | ptb3-revised_charlm       |
| depparse     | combined_charlm           |
| sentiment    | sstplus_charlm            |
| ner          | ontonotes-ww-multi_charlm |

2026-07-16 09:43:03 INFO: Using device: cuda
2026-07-16 09:43:03 INFO: Loading: tokenize
2026-07-16 09:43:04 INFO: Loading: mwt
2026-07-16 09:43:04 INFO: Loading: pos
2026-07-16 09:43:05 INFO: Loading: lemma
2026-07-16 09:43:06 INFO: Loading: constituency
2026-07-16 09:43:06 INFO: Loading: depparse
2026-07-16 09:43:06 INFO: Loading: sentiment
2026-07-16 09:43:06 INFO: Loading: ner


In [4]:
def stanza_filterer(
    recipe_ingredients: List[str], stanza_pipeline: stanza.Pipeline
) -> str:
    """This function converts a list of ingredients into a list of ingredient lemmas
    It is intended to be used via an apply(lambda) until a better way is devised

    Args:
        recipe_ingredients: List[str]

    Returns:
        lemmafied: String
    """
    lemmafied = " ".join(
        str(word.lemma)
        for sent in stanza_pipeline(recipe_ingredients).sentences
        for word in sent.words
        if (
            word.upos not in ["NUM", "DET", "ADV", "CCONJ", "ADP", "SCONJ", "PUNCT"]
            and word is not None
        )
    )
    return lemmafied

In [40]:
test_df = combined_df.sample(n=20)

test_df = test_df.with_columns(
    ingredients_lemmafied=pl.col("ingredients")
    .list.join(
        " brk "
    )  # Converts the raw list of ingredients into a big string with ' brk ' token
    .str.normalize("NFKC")  # Remove accented characters
    .str.to_lowercase()  # lowercase all characters
    .fill_null("Missing ingredients")  # fill nulls with placeholder
    .map_elements(
        lambda x: stanza_filterer(x, nlp)
    )  # Apply the lemmafier function above
)

/tmp/ipykernel_537826/671522762.py:3: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  test_df = test_df.with_columns(


In [41]:
test_df["ingredients_lemmafied"]

ingredients_lemmafied
str
"""cup purpose flour brk cup grah…"
"""vegetable oil need brk cup but…"
"""inch prepare graham cracker cr…"
"""ounce box freeze choppe spinac…"
"""vegetable oil brk tablespoon g…"
…
"""cup choppe peel apple brk cup …"
"""cup ground flaxseed brk cup ca…"
"""cup olive oil brk cup distil w…"


In [42]:
test_df.columns

['mealeon_id',
 'language',
 'source_id',
 'title',
 'origin',
 'ingredients',
 'photo_url',
 'description',
 'steps',
 'cuisines',
 'url',
 'ingredients_lemmafied']

In [43]:
combined_df = combined_df.with_columns(
    ingredients_lemmafied=pl.col("ingredients")
    .list.join(
        " brk "
    )  # Converts the raw list of ingredients into a big string with ' brk ' token
    .str.normalize("NFKC")  # Remove accented characters
    .str.to_lowercase()  # lowercase all characters
    .fill_null("Missing ingredients")  # fill nulls with placeholder
    .map_elements(
        lambda x: stanza_filterer(x, nlp)
    )  # Apply the lemmafier function above
)

/tmp/ipykernel_537826/1866227516.py:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  combined_df = combined_df.with_columns(


In [44]:
combined_df.columns

['mealeon_id',
 'language',
 'source_id',
 'title',
 'origin',
 'ingredients',
 'photo_url',
 'description',
 'steps',
 'cuisines',
 'url',
 'ingredients_lemmafied']

In [45]:
combined_df["ingredients_lemmafied"]

ingredients_lemmafied
str
"""cup unsalted butter chill cube…"
"""cup parmesan cheese brk teaspo…"
"""cup hot water brk cup margarin…"
"""cup white sugar brk cup vegeta…"
"""cup butter brk teaspoon white …"
…
"""tbsp dark brown sugar brk mill…"
"""tbsp groundnut oil vegetable o…"
"""tbsp olive oil brk x kg / lb o…"


In [46]:
# add cleaned dataframe to DVC
combined_df.write_parquet("../data/processed/lemmafied_df.parquet.gzip")
!dvc add "../data/processed/lemmafied_df.parquet.gzip"

 ⠋ Checking graph
Adding...                                                                       
!
                                                                                
!
  0% Checking cache in '/home/awchen/Repos/Projects/MeaLeon/.dvc/cache/files/md5
                                                                                
!
  0%|          |Adding ../data to cache               0/1 [00:00<?,     ?file/s]
                                                                                
!
  0%|          |Checking out /home/awchen/Repos/Projec0/1 [00:00<?,    ?files/s]
100% Adding...|████████████████████████████████████████|1/1 [00:00,  3.64file/s]

To track the changes with git, run:

	git add ../data.dvc

To enable auto staging, run:

	dvc config core.autostage true


In [3]:
# this is a custom function to be used with MLflow to get or create experiments (is from the MLflow team)
def get_mlflow_experiment_id(name):
    # this function allows us to get the experiment ID from an experiment name
    exp = mlflow.get_experiment_by_name(name)
    if exp is None:
        exp_id = mlflow.create_experiment(name)
        return exp_id
    return exp.experiment_id

## Starting DEV stage for TFIDF Encoded model

In [6]:
mlflow.set_tracking_uri(f"https://dagshub.com/{DAGSHUB_USER_NAME}/MeaLeon.mlflow")

# starter idea for making an experiment name, can be the git branch, but need more specificity
experiment_name = f"{DAGSHUB_EMAIL}/DVC-MLflow-integration-test"
mlflow_exp_id = get_mlflow_experiment_id(experiment_name)

# define processed data location and data to be added to DVC
processed_data_base = "../data/processed"
# transformed_recipes_parquet_path = (
#     processed_data_base + "/transformed_recipes.parquet.gzip"
# )
# combined_df_path = processed_data_base + "/lemmafied_df.parquet.gzip"


# define model location
model_directory = "../models/sklearn_model"

# Define the required artifacts associated with the saved custom pyfunc
sklearn_model_path = model_directory + "/python_model.pkl"
sklearn_transformer_path = model_directory + "/sklearn_transformer.pkl"
transformed_recipes_path = model_directory + "/transformed_recipes.pkl"
# combined_df_sample_path = model_directory + "/combined_df_sample.parquet"

artifacts = {
    "sklearn_model": sklearn_model_path,
    "sklearn_transformer": sklearn_transformer_path,
    "transformed_recipes": transformed_recipes_path,
    # "combined_data": combined_df_path,
    # "combined_data_sample": combined_df_sample_path,
}

In [7]:
# Prepare whole dataframe for new processing
!dvc pull

Fetching
!
  0% Checking cache in '/home/awchen/Repos/Projects/MeaLeon/.dvc/cache/files/md5
Fetching                                                                        
Building workspace index                             |79.0 [00:00, 6.98kentry/s]
Comparing indexes                                    |80.0 [00:00, 7.35kentry/s]
Applying changes                                      |2.00 [00:00,   124file/s]
M       ../data/
1 file modified


In [5]:
# this part can be done after a dvc pull
whole_nlp_df = pl.read_parquet(
    "../data/processed/lemmafied_df.parquet.gzip", use_pyarrow=True
)
whole_nlp_df.head(3)

mealeon_id,language,source_id,title,origin,ingredients,photo_url,description,steps,cuisines,url,ingredients_lemmafied
str,str,str,str,str,list[str],str,str,list[str],list[str],str,str
"""AllRecipes-58c6dd19a6a89328e71…","""English""","""6664""","""Basil, Roasted Peppers and Mon…","""AllRecipes""","[""1/2 cup unsalted butter, chilled and cubed"", ""1 cup chopped onion"", … ""1/2 cup chopped fresh basil""]","""http://images.media-allrecipes…","""I just started adding my favor…","[""Preheat oven to 400 degrees F (205 degrees C). Butter a 9x9x2 inch baking pan."", ""Melt 1 tablespoon butter in medium nonstick skillet over medium-low heat. Add onion and saute until tender, about 10 minutes. Cool."", … ""Bake cornbread until golden and tester inserted comes out clean, about 45 minutes. Cool 20 minutes in pan. Cut cornbread into squares.""]","[""Missing Cuisine""]","""http://allrecipes.com/Recipe/6…","""cup unsalted butter chill cube…"
"""AllRecipes-59773cd5496dc7446c4…","""English""","""6663""","""Crispy Cheese Twists""","""AllRecipes""","[""1/2 cup Parmesan cheese"", ""3/4 teaspoon ground black pepper"", … ""1 egg white""]","""http://images.media-allrecipes…","""These are great as an appetize…","[""Combine parmesan cheese, pepper and garlic powder. Unfold pastry sheets onto cutting board. Brush lightly with egg white; sprinkle each sheet with 1/4 of the cheese mixture. Lightly press into pastry, turn over; repeat. Cut each sheet into 12 (1-inch) strips; twist."", ""Place on ungreased cookie sheet and bake in 350 degrees F (175 degrees C) oven for 15 minutes or until golden brown.""]","[""Missing Cuisine""]","""http://allrecipes.com/Recipe/6…","""cup parmesan cheese brk teaspo…"
"""AllRecipes-4022751697d6c412320…","""English""","""6665""","""Mom's Yeast Rolls""","""AllRecipes""","[""2 cups hot water"", ""1/2 cup margarine"", … ""2 eggs""]","""http://images.media-allrecipes…","""This is the best bread recipe.…","[""Melt margarine in hot water. Add sugar and salt and stir. Add cold water and yeast. Stir to dissolve yeast."", ""Add 3 cups flour and mix. Add eggs and 2 1/2 - 3 cups more flour. Mix, cover and let rise until dough doubles in size."", … ""Make walnut size balls of dough. Place about 2 inches apart in well-buttered 9 x 13 inch pan. Bake in a preheated 350 degrees F (175 degrees C) oven for 30-45 minutes. Brush top of rolls with margarine while hot.""]","[""Missing Cuisine""]","""http://allrecipes.com/Recipe/6…","""cup hot water brk cup margarin…"


In [9]:
whole_nlp_df.columns

['mealeon_id',
 'language',
 'source_id',
 'title',
 'origin',
 'ingredients',
 'photo_url',
 'description',
 'steps',
 'cuisines',
 'url',
 'ingredients_lemmafied']

In [ ]:
# load from MLflow
mlflow_client = mlflow.tracking.MlflowClient(
    tracking_uri=f"https://dagshub.com/{DAGSHUB_USER_NAME}/MeaLeon.mlflow"
)

# cv_params are parameters for the sklearn CountVectorizer or TFIDFVectorizer
sklearn_transformer_params = {
    "analyzer": CustomSKLearnAnalyzer().ngram_maker(
        min_ngram_length=1,
        max_ngram_length=4,
    ),
    "min_df": 3,
    "binary": False,
}

# pipeline_params are parameters that will be logged in MLFlow and are a superset of library parameters
pipeline_params = {"stanza_model": "en", "sklearn-transformer": "TFIDF"}

# update the pipeline parameters with the library-specific ones so that they show up in MLflow Tracking
pipeline_params.update(sklearn_transformer_params)

with mlflow.start_run(experiment_id=mlflow_exp_id):
    # LOG PARAMETERS
    mlflow.log_params(pipeline_params)

    # LOG INPUTS (QUERIES) AND OUTPUTS
    # MLflow example uses a list of strings or a list of str->str dicts
    # Will be useful in STAGING/Evaluation

    # LOG MODEL
    # Instantiate sklearn TFIDFVectorizer
    sklearn_transformer = TfidfVectorizer(**sklearn_transformer_params)

    print("\n")
    print("-" * 80)
    print("sklearn fit transform on ingredients:")

    model_input = whole_nlp_df["ingredients_lemmafied"]

    # print("\n")
    # print("-" * 80)
    # print("Input Data: ")
    # print(model_input[0:3])

    print("\n")
    print("-" * 80)
    print("Input Data Shape: ")
    print(model_input.shape)

    random_sample = model_input.sample(3, seed=200)

    print("\n")
    print("-" * 80)
    print("Random 3 Records from Input Data: ")
    print(random_sample)

    # Do fit transform on data
    response = sklearn_transformer.fit_transform(tqdm(model_input))

    print("\n")
    print("-" * 80)
    print("Exporting sklearn transformer: ")
    with open(sklearn_transformer_path, "wb") as fo:
        pickle.dump(sklearn_transformer, fo)
    print("sklearn transformer export done")

    # transformed_recipes = pl.from_numpy(
    #     data=response.toarray(),
    #     schema=sklearn_transformer.get_feature_names_out().tolist()
    # )

    # signature = infer_signature(
    #     model_input=model_input, model_output=response
    # )

    # print("\n")
    # print("-" * 80)
    # print("Transformed Data:")
    # print(transformed_recipes.head())

    # combined_df = pl.concat(
    #     items=[whole_nlp_df, transformed_recipe],
    #     how="horizontal"
    # )
    # combined_df_sample = combined_df.sample(3, seed=200)

    # print("\n")
    # print("-" * 80)
    # print("Random Sample of Combined Data:")
    # print(combined_df_sample.head())

    # transformed_recipes.write_parquet(
    #     file=transformed_recipes_parquet_path, compression="gzip"
    # )

    # combined_df.write_parquet(file=combined_df_path, compression="gzip")

    # combined_df_sample.write_parquet(file=combined_df_sample_path)

    model_info = mlflow.pyfunc.log_model(
        # code_path=["../src/backend/"],
        python_model=CustomSKLearnWrapper(),
        input_example=whole_nlp_df["ingredients_lemmafied"][0],
        # signature=signature,
        artifact_path="sklearn_model",
        artifacts=artifacts,
        registered_model_name="sklearn_model",
    )

    # since this uses a custom Stanza analyzer, we have to use a custom mlflow.Pyfunc.PythonModel



--------------------------------------------------------------------------------
sklearn fit transform on ingredients:


--------------------------------------------------------------------------------
Input Data Shape: 
(141003,)


--------------------------------------------------------------------------------
Random 3 Records from Input Data: 
shape: (3,)
Series: 'ingredients_lemmafied' [str]
[
	"ounce can cherry pie fill divi…
	"tablespoon chile garlic sauce …
	"tablespoon active dry yeast br…
]


100%|██████████| 141003/141003 [00:09<00:00, 14121.24it/s]




--------------------------------------------------------------------------------
Exporting sklearn transformer: 


2026/07/15 15:23:22 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.
2026/07/15 15:23:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


sklearn transformer export done


2026/07/15 15:23:22 INFO mlflow.models.signature: Running the predict function to generate output based on input example
2026/07/15 15:23:22 WARNING mlflow.models.signature: Failed to run the predict function on input example. To see the full traceback, set logging level to DEBUG.
2026/07/15 15:23:22 WARNING mlflow.pyfunc: Provided signature does not match the signature inferred from the Python model's `predict` function type hint. Signature inferred from type hint will be used:
inputs: 
  [string (required)]
outputs: 
  [Any (required)]
params: 
  None

Remove the `signature` parameter or ensure it matches the inferred signature. In a future release, this warning will become an exception, and the signature must align with the type hint.


2026/07/15 15:25:02 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmp79re3uo3/model, flavor: python_function). Fall back to return ['cloudpickle==3.1.1']. Set logging level to DEBUG to see the full traceback. 
2026/07/15 15:25:02 WARNING mlflow.models.model: Failed to validate serving input example {
  "inputs": "cup unsalted butter chill cube brk .... Alternatively, you can avoid passing input example and pass model signature instead when logging the model. To ensure the input example is valid prior to serving, please try calling `mlflow.models.validate_serving_input` on the model uri and serving input example. A serving input example can be generated from model input example using `mlflow.models.convert_input_example_to_serving_input` function.
Got error: No module named 'src.custom_stanza_mlflow'


In [ ]:
mlflow.set_tracking_uri(f"https://dagshub.com/{DAGSHUB_USER_NAME}/MeaLeon.mlflow")

# starter idea for making an experiment name, can be the git branch, but need more specificity
experiment_name = f"{DAGSHUB_EMAIL}/DVC-MLflow-integration-test"
mlflow_exp_id = get_mlflow_experiment_id(experiment_name)

# define processed data location and data to be added to DVC
processed_data_base = "../data/processed"
# transformed_recipes_parquet_path = (
#     processed_data_base + "/transformed_recipes.parquet.gzip"
# )
# combined_df_path = processed_data_base + "/lemmafied_df.parquet.gzip"


# define model location
model_directory = "../models/sklearn_model"

# Define the required artifacts associated with the saved custom pyfunc
# sklearn_model_path = model_directory + "/python_model.pkl"
sklearn_transformer_path = model_directory + "/sklearn_transformer.pkl"
# transformed_recipes_path = model_directory + "/transformed_recipes.pkl"
# combined_df_sample_path = model_directory + "/combined_df_sample.parquet"

artifacts = {
    # "sklearn_model": sklearn_model_path,
    "sklearn_transformer": sklearn_transformer_path,
    # "transformed_recipes": transformed_recipes_path,
    # "combined_data": combined_df_path,
    # "combined_data_sample": combined_df_sample_path,
}

# load from MLflow
mlflow_client = mlflow.tracking.MlflowClient(
    tracking_uri=f"https://dagshub.com/{DAGSHUB_USER_NAME}/MeaLeon.mlflow"
)

with open(sklearn_transformer_path, "rb") as f:
    vectorizer = pickle.load(f)

with mlflow.start_run(run_name="first_tfidf"):
    model_info = mlflow.pyfunc.log_model(
        # code_path=["../src/backend/"],
        python_model=CustomSKLearnWrapper(),
        input_example=whole_nlp_df["ingredients_lemmafied"][0],
        # signature=signature,
        artifact_path="sklearn_transformer",
        artifacts=artifacts,
        registered_model_name="sklearn_transformer",
    )

latest = mlflow_client.get_latest_versions("sklearn_transformer")[0]
mlflow_client.transition_model_version_stage(
    name="sklearn_transformer",
    version=latest.version,
    stage="Production",
)

2026/07/16 19:52:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/16 19:52:41 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


🏃 View run first_tfidf at: https://dagshub.com/AaronWChen/MeaLeon.mlflow/#/experiments/0/runs/8d21158b34cf4852b5d14d18e36f2da8
🧪 View experiment at: https://dagshub.com/AaronWChen/MeaLeon.mlflow/#/experiments/0


MlflowException: `python_model` must be a PythonModel instance, callable object, or path to a script that uses set_model() to set a PythonModel instance or callable object.

In [9]:
mlflow.set_tracking_uri("http://localhost:5001")

# starter idea for making an experiment name, can be the git branch, but need more specificity
experiment_name = f"{DAGSHUB_EMAIL}/DVC-MLflow-integration-test"
mlflow_exp_id = get_mlflow_experiment_id(experiment_name)

# define processed data location and data to be added to DVC
processed_data_base = "../data/processed"
# transformed_recipes_parquet_path = (
#     processed_data_base + "/transformed_recipes.parquet.gzip"
# )
# combined_df_path = processed_data_base + "/lemmafied_df.parquet.gzip"


# define model location
model_directory = "../models/sklearn_model"

# Define the required artifacts associated with the saved custom pyfunc
# sklearn_model_path = model_directory + "/python_model.pkl"
sklearn_transformer_path = model_directory + "/sklearn_transformer.pkl"
# transformed_recipes_path = model_directory + "/transformed_recipes.pkl"
# combined_df_sample_path = model_directory + "/combined_df_sample.parquet"

artifacts = {
    # "sklearn_model": sklearn_model_path,
    "sklearn_transformer": sklearn_transformer_path,
    # "transformed_recipes": transformed_recipes_path,
    # "combined_data": combined_df_path,
    # "combined_data_sample": combined_df_sample_path,
}

# load from MLflow
mlflow_client = mlflow.tracking.MlflowClient(tracking_uri="http://localhost:5001")

with open(sklearn_transformer_path, "rb") as f:
    vectorizer = pickle.load(f)

with mlflow.start_run(run_name="first_tfidf"):
    model_info = mlflow.pyfunc.log_model(
        # code_path=["../src/backend/"],
        python_model=vectorizer,
        input_example=whole_nlp_df["ingredients_lemmafied"][0],
        # signature=signature,
        artifact_path="sklearn_transformer",
        artifacts=artifacts,
        registered_model_name="sklearn_transformer",
    )

latest = mlflow_client.get_latest_versions("sklearn_transformer")[0]
mlflow_client.transition_model_version_stage(
    name="sklearn_transformer",
    version=latest.version,
    stage="Production",
)

MlflowException: API request to http://localhost:5001/api/2.0/mlflow/experiments/get-by-name failed with exception HTTPConnectionPool(host='localhost', port=5001): Max retries exceeded with url: /api/2.0/mlflow/experiments/get-by-name?experiment_name=awc33%40cornell.edu%2FDVC-MLflow-integration-test (Caused by ProtocolError('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer')))

In [ ]:
!dvc add "../data/processed/transformed_recipes.parquet.gzip"

In [ ]:
!dvc add "../data/processed/combined_df.parquet.gzip"